In [21]:
from pymilvus import MilvusClient

client = MilvusClient("./milvus_demo.db", db_name="demo")

# client = MilvusClient(uri="http://localhost:19530", token="username:password")

if client.has_collection(collection_name="demo_collection"):
    client.drop_collection(collection_name="demo_collection")

client.create_collection(
    collection_name="demo_collection",
    dimension=768,  # The vectors we will use in this demo has 768 dimensions
)

# 删除 collections
# client.drop_collection(collection_name="demo_collection")
client.drop_collection()


In [25]:
from langchain_openai import OpenAIEmbeddings
# 超时 —— 无法执行成功
import os


# 配置硅基流动的 API
os.environ['OPENAI_API_KEY'] = os.getenv('SILICON_API_KEY')
os.environ['OPENAI_BASE_URL'] = os.getenv('SILICON_BASE_URL_OPEN_AI')

# 初始化 Embedding 模型
embedding_model = OpenAIEmbeddings(
    model=os.getenv('SILICON_EMBEDDING_MODEL', 'Qwen/Qwen3-VL-Embedding-8B'),
    openai_api_base=os.getenv('SILICON_BASE_URL_OPEN_AI')
)

docs = [
    "Artificial intelligence was founded as an academic discipline in 1956.",
    "Alan Turing was the first person to conduct substantial research in AI.",
    "Born in Maida Vale, London, Turing was raised in southern England.",
]

vectors = embedding_model.embed_documents(docs)
print("Dim:", embedding_model.dim, vectors[0].shape)  # Dim: 768 (768,)

data = [
    {"id": i, "vector": vectors[i], "text": docs[i], "subject": "history"}
    for i in range(len(vectors))
]

print("Data has", len(data), "entities, each with fields: ", data[0].keys())
print("Vector dim:", len(data[0]["vector"]))


TypeError: str expected, not NoneType

In [17]:
# Mock data
import random

docs = [
    "Artificial intelligence was founded as an academic discipline in 1956.",
    "Alan Turing was the first person to conduct substantial research in AI.",
    "Born in Maida Vale, London, Turing was raised in southern England.",
]
vectors = [[random.uniform(-1, 1) for _ in range(768)] for _ in docs]
data = [
    {"id": i, "vector": vectors[i], "text": docs[i], "subject": "history"}
    for i in range(len(vectors))
]

print("Data has", len(data), "entities, each with fields: ", data[0].keys())
print("Vector dim:", len(data[0]["vector"]))



Data has 3 entities, each with fields:  dict_keys(['id', 'vector', 'text', 'subject'])
Vector dim: 768


In [18]:
# 插入数据
res = client.insert(collection_name="demo_collection", data=data)

print(res)

{'insert_count': 3, 'ids': [0, 1, 2]}


In [20]:
# 向量搜索
from pymilvus import model

embedding_fn = model.DefaultEmbeddingFunction()

query_vectors = embedding_fn.encode_queries(["Who is Alan Turing?"])

res = client.search(
    collection_name="demo_collection",  # target collection
    data=query_vectors,  # query vectors
    limit=2,  # number of returned entities
    output_fields=["text", "subject"],  # specifies fields to be returned
)

print(res)


I0625 22:34:19.397506 2446082 chttp2_transport.cc:1376] ipv4:127.0.0.1:60554: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11}
E0625 22:34:19.397605 2446082 chttp2_transport.cc:1408] ipv4:127.0.0.1:60554: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 20000ms
'timed out' thrown while requesting HEAD https://huggingface.co/GPTCache/paraphrase-albert-small-v2/resolve/main/config.json
Retrying in 1s [Retry 1/5].


KeyboardInterrupt: 